# Finite Element Analysis: From Geometric Intuition to Python Implementation

## Introduction: Why FEA?

Imagine you're an engineer trying to predict how heat flows through a complex metal part, or how a bridge deforms under load. These problems involve solving partial differential equations (PDEs) on irregular geometries - exactly where traditional analytical methods fail.

Finite Element Analysis (FEA) provides a systematic way to:
1. **Break complex problems into simple pieces** (meshing)
2. **Approximate solutions locally** using simple functions (shape functions)
3. **Connect pieces together** to form a global system (assembly)
4. **Solve the resulting algebraic system** (linear algebra)

This notebook builds intuition for each step, progressing from geometric concepts to complete Python implementations.

## Learning Objectives

By the end of this notebook, you will:
- Understand shape functions as geometric objects
- Visualize the weak form and weighted residuals
- Implement a complete 1D FEA solver in Python
- Extend concepts to 2D triangular elements
- Build intuition for matrix assembly and boundary conditions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
from mpl_toolkits.mplot3d import Axes3D
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

# Set up nice plotting parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

## 1. Shape Functions: The Building Blocks of FEA

### Geometric Intuition

Shape functions are the heart of FEA. Think of them as **weight functions** that tell us how much each node contributes to the solution at any point within an element.

For a triangle, the shape functions are simply the **barycentric coordinates** - these are the most natural way to describe position within a triangle:

In [ ]:
def plot_shape_functions():
    """
    Visualize shape functions as 3D pyramids over a triangular element.
    This builds geometric intuition for how shape functions work.
    """
    # Triangle vertices (any non-degenerate triangle works)
    x = np.array([0.0, 1.0, 0.2])
    y = np.array([0.0, 0.0, 1.0])

    # Create grid points inside triangle using barycentric coordinates
    N = 50
    L1 = np.linspace(0, 1, N)
    L2 = np.linspace(0, 1, N)
    L1, L2 = np.meshgrid(L1, L2)
    L3 = 1 - L1 - L2

    # Keep only points inside triangle (where all λi >= 0)
    mask = (L1 >= 0) & (L2 >= 0) & (L3 >= 0)
    L1, L2, L3 = L1[mask], L2[mask], L3[mask]

    # Convert barycentric → Cartesian coordinates
    X = L1 * x[0] + L2 * x[1] + L3 * x[2]
    Y = L1 * y[0] + L2 * y[1] + L3 * y[2]

    # Shape functions ARE the barycentric coordinates themselves!
    N1, N2, N3 = L1, L2, L3

    # Plot each as a 3D pyramid
    fig = plt.figure(figsize=(15, 4))

    for i, (Ni, title) in enumerate(zip([N1, N2, N3], [r"$N_1$", r"$N_2$", r"$N_3$"]), 1):
        ax = fig.add_subplot(1, 3, i, projection='3d')
        ax.plot_trisurf(X, Y, Ni, cmap='viridis', linewidth=0.2, antialiased=True)
        ax.set_title(f"Shape function {title}", fontsize=14)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("N")
        ax.set_zlim(0, 1)
        ax.view_init(elev=30, azim=-135)
        
        # Add contour lines at base
        ax.contour(X, Y, Ni, levels=[0.1, 0.3, 0.5, 0.7, 0.9], 
                  offset=0, cmap='viridis', alpha=0.3, linewidths=0.5)

    plt.tight_layout()
    plt.show()
    
    print("Key Properties of Shape Functions:")
    print("1. Ni = 1 at node i, Ni = 0 at other nodes")
    print("2. Sum(Ni) = 1 everywhere (partition of unity)")
    print("3. Ni are linear within triangles")
    print("4. These are just barycentric coordinates!")

plot_shape_functions()

### Mathematical Foundation

For a triangle with vertices $(x_1, y_1)$, $(x_2, y_2)$, $(x_3, y_3)$, the barycentric coordinates are:

$$\begin{bmatrix} N_1 \\ N_2 \\ N_3 \end{bmatrix} = \frac{1}{2A} \begin{bmatrix} 
y_2 - y_3 & x_3 - x_2 \\
y_3 - y_1 & x_1 - x_3 \\
y_1 - y_2 & x_2 - x_1 
\end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} + \frac{1}{2A} \begin{bmatrix} 
x_2 y_3 - x_3 y_2 \\
x_3 y_1 - x_1 y_3 \\
x_1 y_2 - x_2 y_1 
\end{bmatrix}$$

where $A$ is the triangle area.

This linear relationship is crucial - it makes FEA computations tractable!

## 2. The Weak Form: Why We Use Weighted Residuals

### From Strong to Weak Form

Consider the Poisson equation: $-\nabla^2 u = f$ in domain $\Omega$ with boundary conditions.

The **strong form** requires the equation to hold at every point. This is restrictive!

The **weak form** relaxes this requirement: instead of requiring the equation to hold pointwise, we require it to hold **on average** when weighted by test functions.

### Physical Intuition

Think of weighted residuals as a **smoothing operation**:
- If the residual $r = -\nabla^2 u - f$ is not zero everywhere, but its weighted average is zero, the solution is still "good enough"
- Different weight functions test different aspects of the solution
- Galerkin's method: use shape functions as weight functions

In [ ]:
def plot_weighted_residuals():
    """
    Visualize residual, weighting function, and weighted residual.
    This builds intuition for the weak form concept.
    """
    # Define triangle vertices
    x = np.array([0.0, 1.0, 0.2])
    y = np.array([0.0, 0.0, 1.0])

    # Create grid points inside the triangle using barycentric coordinates
    N = 50
    L1 = np.linspace(0, 1, N)
    L2 = np.linspace(0, 1, N)
    L1, L2 = np.meshgrid(L1, L2)
    L3 = 1 - L1 - L2
    mask = (L1 >= 0) & (L2 >= 0) & (L3 >= 0)
    L1, L2, L3 = L1[mask], L2[mask], L3[mask]

    # Cartesian coordinates
    X = L1 * x[0] + L2 * x[1] + L3 * x[2]
    Y = L1 * y[0] + L2 * y[1] + L3 * y[2]

    # Define an artificial "residual" r(x,y) = PDE not satisfied perfectly
    # Use a smooth function for demonstration
    R = np.sin(np.pi * X) * np.sin(np.pi * Y)

    # Shape functions (weights)
    N1, N2, N3 = L1, L2, L3

    # Compute weighted residuals
    WR1 = R * N1
    WR2 = R * N2
    WR3 = R * N3

    # Plotting
    fig = plt.figure(figsize=(18, 5))

    def plot_surface(ax, X, Y, Z, title):
        ax.plot_trisurf(X, Y, Z, cmap='viridis', linewidth=0.2, antialiased=True)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_zlabel('value')
        ax.set_title(title, fontsize=14)
        ax.view_init(elev=30, azim=-135)

    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    plot_surface(ax1, X, Y, R, 'Residual r(x,y)')

    ax2 = fig.add_subplot(1, 3, 2, projection='3d')
    plot_surface(ax2, X, Y, N1, 'Weighting function N₁(x,y)')

    ax3 = fig.add_subplot(1, 3, 3, projection='3d')
    plot_surface(ax3, X, Y, WR1, 'Weighted Residual r×N₁ (Weak Form)')

    plt.tight_layout()
    plt.show()
    
    print("Weak Form Interpretation:")
    print("∫ r(x,y) × N₁(x,y) dA = 0")
    print("This means: 'The residual, when weighted by N₁, averages to zero'")
    print("We have 3 such equations (one for each shape function)")

plot_weighted_residuals()

## 3. 1D FEA Implementation: Step-by-Step

Let's implement a complete 1D FEA solver for the boundary value problem:

$$-\frac{d^2u}{dx^2} = f(x), \quad u(0) = u(1) = 0$$

### Step 1: Problem Setup

We'll solve this on the domain $[0,1]$ with linear elements.

In [ ]:
def setup_1d_problem(n_elements=5):
    """
    Set up 1D FEA problem with linear elements.
    """
    # Domain [0,1] divided into n_elements
    n_nodes = n_elements + 1
    nodes = np.linspace(0, 1, n_nodes)
    
    # Element connectivity: each element connects two adjacent nodes
    elements = np.array([[i, i+1] for i in range(n_elements)])
    
    return nodes, elements

# Example setup
nodes, elements = setup_1d_problem(5)
print(f"Nodes: {nodes}")
print(f"Elements:\n{elements}")
print(f"Number of elements: {len(elements)}")
print(f"Number of nodes: {len(nodes)}")

### Step 2: Shape Functions for 1D Linear Elements

For a 1D linear element with nodes at $x_i$ and $x_{i+1}$, the shape functions are:

$$N_1(x) = \frac{x_{i+1} - x}{h}, \quad N_2(x) = \frac{x - x_i}{h}$$

where $h = x_{i+1} - x_i$ is the element length.

In [ ]:
def linear_shape_functions_1d(x, x_left, x_right):
    """
    1D linear shape functions.
    Returns N1, N2 at position x within element [x_left, x_right]
    """
    h = x_right - x_left
    N1 = (x_right - x) / h  # Weight for left node
    N2 = (x - x_left) / h   # Weight for right node
    return N1, N2

# Visualize shape functions for an element
x_left, x_right = 0.3, 0.5
x_elem = np.linspace(x_left, x_right, 100)
N1_vals, N2_vals = [], []

for x in x_elem:
    N1, N2 = linear_shape_functions_1d(x, x_left, x_right)
    N1_vals.append(N1)
    N2_vals.append(N2)

plt.figure(figsize=(10, 4))
plt.plot(x_elem, N1_vals, 'b-', linewidth=2, label='N₁(x) (left node)')
plt.plot(x_elem, N2_vals, 'r-', linewidth=2, label='N₂(x) (right node)')
plt.plot([x_left, x_right], [1, 1], 'k--', alpha=0.3)
plt.xlabel('x')
plt.ylabel('Shape function value')
plt.title('1D Linear Shape Functions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim([-0.1, 1.1])
plt.show()

print("Properties:")
print(f"N₁({x_left}) = {linear_shape_functions_1d(x_left, x_left, x_right)[0]:.1f}, N₁({x_right}) = {linear_shape_functions_1d(x_right, x_left, x_right)[0]:.1f}")
print(f"N₂({x_left}) = {linear_shape_functions_1d(x_left, x_left, x_right)[1]:.1f}, N₂({x_right}) = {linear_shape_functions_1d(x_right, x_left, x_right)[1]:.1f}")
print(f"N₁ + N₂ = {linear_shape_functions_1d(x_left + 0.1, x_left, x_right)[0] + linear_shape_functions_1d(x_left + 0.1, x_left, x_right)[1]:.1f} (partition of unity)")

### Step 3: Element Stiffness Matrix

For our 1D problem $-\frac{d^2u}{dx^2} = f$, the element stiffness matrix is:

$$K^e = \int_{x_i}^{x_{i+1}} \frac{dN^T}{dx} \frac{dN}{dx} dx$$

For linear elements, this evaluates to:

$$K^e = \frac{1}{h} \begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}$$

In [ ]:
def element_stiffness_1d(x_left, x_right):
    """
    Compute element stiffness matrix for 1D linear element.
    Problem: -d²u/dx² = f
    """
    h = x_right - x_left
    Ke = (1/h) * np.array([[1, -1], [-1, 1]])
    return Ke

# Test element stiffness matrix
x_left, x_right = 0.3, 0.5
Ke_test = element_stiffness_1d(x_left, x_right)
print("Element stiffness matrix:")
print(Ke_test)
print(f"\nElement length h = {x_right - x_left:.2f}")
print("Interpretation:")
print("- Diagonal: positive stiffness (resistance to change)")
print("- Off-diagonal: negative coupling between nodes")
print("- Smaller elements (smaller h) → larger stiffness values")

### Step 4: Element Load Vector

The element load vector comes from the source term $f(x)$:

$$f^e = \int_{x_i}^{x_{i+1}} N^T f(x) dx$$

For simplicity, let's use $f(x) = 1$ (constant source term):

In [ ]:
def element_load_1d(x_left, x_right, f_func=lambda x: 1):
    """
    Compute element load vector using numerical integration.
    """
    # Simple numerical integration using midpoint rule
    x_mid = (x_left + x_right) / 2
    h = x_right - x_left
    
    # For linear elements with constant f, analytical solution exists:
    # ∫ N₁ f dx = f × h/2, ∫ N₂ f dx = f × h/2
    f_mid = f_func(x_mid)
    fe = np.array([f_mid * h/2, f_mid * h/2])
    
    return fe

# Test element load vector
fe_test = element_load_1d(x_left, x_right)
print("Element load vector:")
print(fe_test)
print("\nInterpretation:")
print("- Each node gets half the total load")
print("- Total load = f × element length")
print(f"- Total load = {1} × {x_right - x_left:.2f} = {(x_right - x_left):.2f}")
print(f"- Sum of load vector = {np.sum(fe_test):.2f} ✓")

### Step 5: Global Assembly

Now we assemble all element matrices into the global system. This is like building a big puzzle where each element contributes to its local neighborhood.

In [ ]:
def assemble_global_system_1d(nodes, elements, f_func=lambda x: 1):
    """
    Assemble global stiffness matrix and load vector.
    """
    n_nodes = len(nodes)
    K = lil_matrix((n_nodes, n_nodes))  # Use sparse format
    f = np.zeros(n_nodes)
    
    for elem_idx, elem in enumerate(elements):
        node1, node2 = elem
        x1, x2 = nodes[node1], nodes[node2]
        
        # Element matrices
        Ke = element_stiffness_1d(x1, x2)
        fe = element_load_1d(x1, x2, f_func)
        
        # Assembly: add element contribution to global system
        for i in range(2):
            for j in range(2):
                K[elem[i], elem[j]] += Ke[i, j]
            f[elem[i]] += fe[i]
    
    return K.tocsr(), f

# Test assembly
nodes, elements = setup_1d_problem(4)
K_global, f_global = assemble_global_system_1d(nodes, elements)

print("Global stiffness matrix:")
print(K_global.toarray())
print("\nGlobal load vector:")
print(f_global)

# Visualize sparsity pattern
plt.figure(figsize=(8, 3))
plt.spy(K_global, markersize=10, color='red')
plt.title('Sparsity Pattern of Global Stiffness Matrix')
plt.xlabel('Node index')
plt.ylabel('Node index')
plt.show()

print("\nObservations:")
print("- Matrix is tridiagonal (each node only connected to neighbors)")
print("- This is typical for 1D problems")
print("- Sparse structure makes solving efficient")

### Step 6: Apply Boundary Conditions

We have Dirichlet boundary conditions: $u(0) = u(1) = 0$.

Implementation strategy: modify the linear system to enforce these values.

In [ ]:
def apply_boundary_conditions_1d(K, f, nodes, bc_dict):
    """
    Apply Dirichlet boundary conditions.
    bc_dict: {node_index: prescribed_value}
    """
    n_nodes = len(nodes)
    K_modified = K.copy()
    f_modified = f.copy()
    
    # Identify free and fixed nodes
    fixed_nodes = list(bc_dict.keys())
    free_nodes = [i for i in range(n_nodes) if i not in fixed_nodes]
    
    # Apply Dirichlet BCs using penalty method
    penalty = 1e10
    for node, value in bc_dict.items():
        K_modified[node, node] += penalty
        f_modified[node] += penalty * value
    
    return K_modified, f_modified, free_nodes

# Apply boundary conditions
bc_dict = {0: 0, len(nodes)-1: 0}  # u(0) = 0, u(1) = 0
K_bc, f_bc, free_nodes = apply_boundary_conditions_1d(K_global, f_global, nodes, bc_dict)

print("Modified stiffness matrix (with BCs):")
print(K_bc.toarray())
print("\nModified load vector (with BCs):")
print(f_bc)
print(f"\nFree nodes: {free_nodes}")
print("Fixed nodes (boundary): [0, 4]")

### Step 7: Solve the System

Finally, we solve the linear system $Ku = f$ to get the nodal values.

In [ ]:
def solve_fea_1d(K, f, nodes, bc_dict):
    """
    Solve the complete 1D FEA problem.
    """
    # Apply boundary conditions
    K_bc, f_bc, free_nodes = apply_boundary_conditions_1d(K, f, nodes, bc_dict)
    
    # Solve system
    u = spsolve(K_bc, f_bc)
    
    return u

# Solve our problem
u_solution = solve_fea_1d(K_global, f_global, nodes, bc_dict)

print("FEA solution:")
for i, (x, u_val) in enumerate(zip(nodes, u_solution)):
    print(f"Node {i}: x = {x:.2f}, u = {u_val:.4f}")

# Compare with analytical solution for -d²u/dx² = 1, u(0)=u(1)=0
# Analytical: u(x) = 0.5 * x * (1 - x)
x_fine = np.linspace(0, 1, 100)
u_analytical = 0.5 * x_fine * (1 - x_fine)

# Plot comparison
plt.figure(figsize=(10, 6))
plt.plot(x_fine, u_analytical, 'r-', linewidth=2, label='Analytical solution')
plt.plot(nodes, u_solution, 'bo-', linewidth=2, markersize=8, label='FEA solution')
plt.xlabel('x')
plt.ylabel('u(x)')
plt.title('1D FEA vs Analytical Solution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compute error
u_interp = np.interp(x_fine, nodes, u_solution)
error = np.abs(u_interp - u_analytical)
max_error = np.max(error)
print(f"\nMaximum error: {max_error:.6f}")
print(f"Relative error: {max_error/np.max(u_analytical):.4%}")

### Step 8: Convergence Study

Let's see how the error decreases as we refine the mesh.

In [ ]:
def convergence_study():
    """
    Study convergence as mesh is refined.
    """
    element_counts = [4, 8, 16, 32, 64]
    errors = []
    
    for n_elem in element_counts:
        # Setup and solve
        nodes, elements = setup_1d_problem(n_elem)
        K, f = assemble_global_system_1d(nodes, elements)
        bc_dict = {0: 0, len(nodes)-1: 0}
        u = solve_fea_1d(K, f, nodes, bc_dict)
        
        # Compute error at nodes
        u_analytical_nodes = 0.5 * nodes * (1 - nodes)
        error = np.max(np.abs(u - u_analytical_nodes))
        errors.append(error)
        
        print(f"Elements: {n_elem:2d}, Max error: {error:.6e}")
    
    # Plot convergence
    plt.figure(figsize=(10, 6))
    plt.loglog(element_counts, errors, 'bo-', linewidth=2, markersize=8, label='FEA error')
    
    # Reference line for O(h²) convergence
    h_values = 1.0 / np.array(element_counts)
    ref_line = errors[0] * (h_values / h_values[0])**2
    plt.loglog(element_counts, ref_line, 'r--', linewidth=2, label='O(h²) reference')
    
    plt.xlabel('Number of elements')
    plt.ylabel('Maximum error')
    plt.title('Convergence Study')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\nConvergence analysis:")
    print("- Error decreases quadratically with element size")
    print("- This is expected for linear elements (second-order convergence)")

convergence_study()

## 4. Extension to 2D: Triangular Elements

Now let's extend our concepts to 2D problems using triangular elements. We'll solve:

$$-\nabla^2 u = f \text{ in } \Omega$$

### 2D Shape Functions

As we saw earlier, 2D triangular shape functions are barycentric coordinates.

In [ ]:
def triangular_element_stiffness(coords):
    """
    Compute element stiffness matrix for triangular element.
    coords: array of shape (3, 2) with vertex coordinates
    """
    x = coords[:, 0]
    y = coords[:, 1]
    
    # Area of triangle using determinant formula
    A = 0.5 * np.abs(np.linalg.det(np.array([
        [1, x[0], y[0]],
        [1, x[1], y[1]],
        [1, x[2], y[2]]
    ])))
    
    # Gradients of shape functions (constant for linear triangles)
    b = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]])
    c = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]])
    B = np.array([b, c]) / (2 * A)
    
    # Element stiffness matrix: Ke = A * B^T * B
    Ke = A * (B.T @ B)
    
    return Ke, A, B

# Test with a simple triangle
triangle_coords = np.array([
    [0.0, 0.0],  # vertex 1
    [1.0, 0.0],  # vertex 2  
    [0.0, 1.0]   # vertex 3
])

Ke_triangle, area, B = triangular_element_stiffness(triangle_coords)

print(f"Triangle area: {area:.3f}")
print("\nGradient matrix B:")
print(B)
print("\nElement stiffness matrix:")
print(Ke_triangle)

print("\nInterpretation:")
print("- B contains gradients of shape functions")
print("- Ke represents element's contribution to global stiffness")
print("- All entries are positive (different from 1D case)")

### 2D Mesh Generation

Let's create a simple 2D mesh and solve a problem.

In [ ]:
from scipy.spatial import Delaunay

def create_rectangular_mesh(nx, ny, width=1.0, height=1.0):
    """
    Create a structured rectangular mesh.
    """
    # Create grid of points
    x = np.linspace(0, width, nx)
    y = np.linspace(0, height, ny)
    X, Y = np.meshgrid(x, y)
    
    # Flatten to get node coordinates
    nodes = np.column_stack([X.ravel(), Y.ravel()])
    
    # Create triangulation
    tri = Delaunay(nodes)
    elements = tri.simplices
    
    return nodes, elements

# Create a simple mesh
nodes_2d, elements_2d = create_rectangular_mesh(6, 6)

print(f"Number of nodes: {len(nodes_2d)}")
print(f"Number of elements: {len(elements_2d)}")

# Visualize mesh
plt.figure(figsize=(8, 8))
plt.triplot(nodes_2d[:, 0], nodes_2d[:, 1], elements_2d, 'k-', linewidth=0.5, alpha=0.6)
plt.plot(nodes_2d[:, 0], nodes_2d[:, 1], 'ro', markersize=3)
plt.xlabel('x')
plt.ylabel('y')
plt.title('2D Triangular Mesh')
plt.gca().set_aspect('equal')
plt.show()

print("\nMesh quality:")
print("- Structured mesh ensures good element quality")
print("- All elements are well-shaped triangles")
print("- Boundary is clearly defined")

### 2D Assembly and Solution

Now let's solve a 2D problem: find the temperature distribution in a square domain with a heat source.

In [ ]:
def solve_2d_poisson(nodes, elements, source_func=lambda x, y: 1):
    """
    Solve 2D Poisson equation: -∇²u = f
    """
    n_nodes = len(nodes)
    K = lil_matrix((n_nodes, n_nodes))
    f = np.zeros(n_nodes)
    
    # Assembly loop
    for elem_idx, elem in enumerate(elements):
        # Get element coordinates
        coords = nodes[elem]
        
        # Element stiffness matrix
        Ke, area, B = triangular_element_stiffness(coords)
        
        # Element load vector (assume constant source)
        centroid = np.mean(coords, axis=0)
        source_val = source_func(centroid[0], centroid[1])
        fe = np.ones(3) * source_val * area / 3.0
        
        # Assemble
        for i in range(3):
            for j in range(3):
                K[elem[i], elem[j]] += Ke[i, j]
            f[elem[i]] += fe[i]
    
    return K.tocsr(), f

# Solve the 2D problem
K_2d, f_2d = solve_2d_poisson(nodes_2d, elements_2d)

print("Assembled 2D system:")
print(f"Global stiffness matrix shape: {K_2d.shape}")
print(f"Number of non-zero entries: {K_2d.nnz}")
print(f"Sparsity: {K_2d.nnz / (K_2d.shape[0] * K_2d.shape[1]):.4f}")

# Visualize sparsity pattern
plt.figure(figsize=(8, 6))
plt.spy(K_2d, markersize=1)
plt.title('2D Stiffness Matrix Sparsity Pattern')
plt.xlabel('Node index')
plt.ylabel('Node index')
plt.show()

In [ ]:
def apply_2d_boundary_conditions(K, f, nodes, tolerance=1e-10):
    """
    Apply boundary conditions: u = 0 on all boundaries.
    """
    n_nodes = len(nodes)
    
    # Identify boundary nodes
    boundary_nodes = []
    
    # Check each node
    for i, (x, y) in enumerate(nodes):
        if (abs(x) < tolerance or abs(x - 1.0) < tolerance or 
            abs(y) < tolerance or abs(y - 1.0) < tolerance):
            boundary_nodes.append(i)
    
    # Apply Dirichlet BCs
    K_bc = K.copy()
    f_bc = f.copy()
    
    penalty = 1e10
    for node in boundary_nodes:
        K_bc[node, node] += penalty
        # f_bc[node] += penalty * 0  # u = 0, so no addition needed
    
    free_nodes = [i for i in range(n_nodes) if i not in boundary_nodes]
    
    return K_bc, f_bc, free_nodes, boundary_nodes

# Apply boundary conditions and solve
K_bc_2d, f_bc_2d, free_nodes_2d, boundary_nodes_2d = apply_2d_boundary_conditions(K_2d, f_2d, nodes_2d)

print(f"Boundary nodes: {len(boundary_nodes_2d)}")
print(f"Free nodes: {len(free_nodes_2d)}")

# Solve the system
u_2d = spsolve(K_bc_2d, f_bc_2d)

print(f"\nSolution statistics:")
print(f"Min value: {np.min(u_2d):.4f}")
print(f"Max value: {np.max(u_2d):.4f}")
print(f"Mean value: {np.mean(u_2d):.4f}")

In [ ]:
# Visualize the 2D solution
plt.figure(figsize=(12, 5))

# Plot 1: Solution contour
plt.subplot(1, 2, 1)
contour = plt.tricontourf(nodes_2d[:, 0], nodes_2d[:, 1], elements_2d, u_2d, levels=20, cmap='viridis')
plt.colorbar(contour, label='u(x,y)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Solution Contours')
plt.gca().set_aspect('equal')

# Plot 2: 3D surface
ax = plt.subplot(1, 2, 2, projection='3d')
surf = ax.plot_trisurf(nodes_2d[:, 0], nodes_2d[:, 1], u_2d, 
                       cmap='viridis', alpha=0.8, linewidth=0.1)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('u(x,y)')
ax.set_title('3D Solution Surface')
ax.view_init(elev=30, azim=45)

plt.tight_layout()
plt.show()

print("Physical interpretation:")
print("- This represents steady-state temperature distribution")
print("- Boundary is held at zero temperature")
print("- Heat source is uniform throughout the domain")
print("- Maximum temperature occurs at the center")
print("- Solution is symmetric due to symmetric boundary conditions")

## 5. Key Takeaways

### What We Learned

1. **Shape Functions as Geometric Objects**: Shape functions are fundamentally about weighting - they tell us how much each node contributes to the solution at any point.

2. **Weak Form Intuition**: The weak form relaxes the requirement that PDEs hold pointwise, instead requiring them to hold "on average" when weighted by test functions.

3. **Assembly Process**: Global systems are built by adding local element contributions - each element only affects its local neighborhood.

4. **Boundary Conditions**: These are essential for well-posed problems and significantly affect the solution structure.

5. **Convergence**: As we refine the mesh, FEA solutions converge to the true solution, with predictable error behavior.

### Computational Aspects

- **Sparse Matrices**: FEA naturally produces sparse systems, making large-scale problems tractable
- **Local Operations**: Most computations (element matrices) are local and parallelizable
- **Linear Algebra**: Everything reduces to solving linear systems $Ku = f$

### Physical Interpretation

FEA is not just abstract mathematics - it represents real physical behavior:
- Stiffness matrices represent physical resistance to deformation/flow
- Load vectors represent external forces/sources
- Solutions represent physical quantities (temperature, displacement, etc.)

### Next Steps

This foundation prepares you for:
- More complex element types (quadratic, 3D elements)
- Advanced boundary conditions (Neumann, Robin)
- Time-dependent problems
- Nonlinear problems
- Real-world engineering applications

The geometric intuition and systematic approach you've learned here extends to all of FEA!